# Milestone 1 — Data Preprocessing
### AI-Powered Soil Analytics System for Nutrient Assessment and Intelligent Crop Advisory

This notebook covers Week 2 of Milestone 1: image preprocessing (cleaning, resizing, normalization, background handling, augmentation, train/val/test split) and structured soil-test data cleaning (missing values, invalid values, outliers, encoding, scaling, EDA).

**Dataset status:**
- **Structured data:** uses the real acquired dataset, `data_core.csv` (8,000 rows — Temperature, Humidity, Moisture, Soil Type, Crop Type, Nitrogen, Potassium, Phosphorous, Fertilizer Name). Upload `data_core.csv` to the Colab file browser (or mount Drive) before running Section 4.
- **Image data:** not yet sourced. The image pipeline (Section 3) still runs on small placeholder images generated in-notebook, just to validate the pipeline code — re-run it against a real soil-image dataset once one is selected.
- Note: this dataset does not include pH or Organic matter, so those two parameters aren't covered by the cleaning/EDA below.

## 1. Setup & Imports

In [ ]:
!pip install opencv-python-headless pandas numpy scikit-learn matplotlib seaborn Pillow -q

import os
import cv2
import shutil
import hashlib
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Optional: mount Google Drive if your dataset lives there
# from google.colab import drive
# drive.mount('/content/drive')

RAW_IMG_DIR = "data/raw/images"
PROCESSED_DIR = "data/processed"
CSV_PATH = "data_core.csv"   # upload this file to the Colab session (or set a Drive path)

os.makedirs(RAW_IMG_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)


## 2. Placeholder Image Generator
No real soil-image dataset has been sourced yet, so this cell creates small placeholder images purely to validate that the image pipeline (Section 3) runs correctly end-to-end. **Delete this cell once a real labeled soil-image dataset is available**, and point `RAW_IMG_DIR` at it instead.

In [ ]:
def make_sample_images(n=30, out_dir=RAW_IMG_DIR):
    classes = ["alluvial", "black", "red", "laterite"]
    for c in classes:
        os.makedirs(f"{out_dir}/{c}", exist_ok=True)
    for i in range(n):
        c = random.choice(classes)
        img = np.random.randint(0, 255, (300, 300, 3), dtype=np.uint8)
        cv2.imwrite(f"{out_dir}/{c}/img_{i:03d}.jpg", img)
    print(f"Created {n} placeholder images across {len(classes)} classes (demo only — not real soil imagery).")

make_sample_images()


## 3. Image Preprocessing Pipeline (placeholder data — pending real image dataset)

In [ ]:
IMG_SIZE = (224, 224)  # matches ResNet-50 / EfficientNet input

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def remove_corrupted_and_duplicates(img_dir):
    seen_hashes = {}
    removed = 0
    for root, _, files in os.walk(img_dir):
        for fname in files:
            fpath = os.path.join(root, fname)
            img = cv2.imread(fpath)
            if img is None:
                os.remove(fpath)          # corrupted
                removed += 1
                continue
            h = file_hash(fpath)
            if h in seen_hashes:
                os.remove(fpath)          # duplicate
                removed += 1
            else:
                seen_hashes[h] = fpath
    print(f"Removed {removed} corrupted/duplicate images.")

def remove_background(img, blur_ksize=5):
    """Simple background isolation using Otsu thresholding."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (blur_ksize, blur_ksize), 0)
    _, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    result = cv2.bitwise_and(img, img, mask=mask)
    return result

def preprocess_image(fpath, size=IMG_SIZE):
    img = cv2.imread(fpath)
    if img is None:
        return None
    img = remove_background(img)
    img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0     # normalize to [0,1]
    return img

def augment_image(img):
    """Returns a list of augmented variants: original + flip + brightness jitter + rotation."""
    variants = [img]
    variants.append(cv2.flip(img, 1))                              # horizontal flip
    bright = np.clip(img * random.uniform(0.8, 1.2), 0, 1)          # brightness jitter
    variants.append(bright)
    angle = random.uniform(-15, 15)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(img, M, (w, h))
    variants.append(rotated)
    return variants

def build_image_dataset(raw_dir, processed_dir, augment=True, val_size=0.15, test_size=0.15):
    remove_corrupted_and_duplicates(raw_dir)

    records = []  # (image_array, label)
    for class_name in os.listdir(raw_dir):
        class_dir = os.path.join(raw_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            fpath = os.path.join(class_dir, fname)
            img = preprocess_image(fpath)
            if img is None:
                continue
            variants = augment_image(img) if augment else [img]
            for v in variants:
                records.append((v, class_name))

    X = [r[0] for r in records]
    y = [r[1] for r in records]

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=(val_size + test_size), stratify=y, random_state=42)
    rel_test = test_size / (val_size + test_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=rel_test, stratify=y_temp, random_state=42)

    for split_name, X_split, y_split in [("train", X_train, y_train),
                                          ("val", X_val, y_val),
                                          ("test", X_test, y_test)]:
        for i, (img, label) in enumerate(zip(X_split, y_split)):
            out_dir = os.path.join(processed_dir, split_name, label)
            os.makedirs(out_dir, exist_ok=True)
            out_img = (img * 255).astype(np.uint8)
            out_img_bgr = cv2.cvtColor(out_img, cv2.COLOR_RGB2BGR)
            cv2.imwrite(os.path.join(out_dir, f"{split_name}_{i:04d}.jpg"), out_img_bgr)

    print(f"Dataset built: {len(X_train)} train / {len(X_val)} val / {len(X_test)} test images")
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = build_image_dataset(RAW_IMG_DIR, PROCESSED_DIR)


## 4. Structured Data Cleaning (Pandas) — real dataset: `data_core.csv`
Upload `data_core.csv` to this Colab session's file browser (left sidebar -> Files -> upload) before running this cell, or set `CSV_PATH` to a Drive path.

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Initial shape:", df.shape)
print(df.dtypes)
print("--- missing values ---")
print(df.isna().sum())
print("--- duplicate rows ---")
print(df.duplicated().sum())

numeric_cols = ["Temparature", "Humidity", "Moisture", "Nitrogen", "Potassium", "Phosphorous"]
categorical_cols = ["Soil Type", "Crop Type", "Fertilizer Name"]

# 1. Remove exact duplicate rows (none expected, but kept for robustness)
df = df.drop_duplicates()

# 2. Handle missing values (median imputation for numeric columns, if any appear)
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# 3. Invalid-value checks (domain-specific bounds — tighten as needed)
valid_ranges = {
    "Temparature": (0, 60),     # deg C
    "Humidity": (0, 100),       # %
    "Moisture": (0, 100),       # %
    "Nitrogen": (0, 1000),
    "Potassium": (0, 1000),
    "Phosphorous": (0, 1000),
}
for col, (low, high) in valid_ranges.items():
    invalid_mask = ~df[col].between(low, high)
    df.loc[invalid_mask, col] = df[col].median()
    print(f"{col}: fixed {invalid_mask.sum()} invalid values")

# 4. Outlier detection (IQR method) — flagged, not auto-dropped
def flag_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ~series.between(lower, upper)

for col in numeric_cols:
    df[f"{col}_outlier"] = flag_outliers_iqr(df[col])
    print(f"{col}: {df[f'{col}_outlier'].sum()} outliers flagged")

# 5. Encode categorical columns
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[f"{col}_encoded"] = le.fit_transform(df[col])
    encoders[col] = le

# 6. Scale numeric features
scaler = StandardScaler()
df[[f"{c}_scaled" for c in numeric_cols]] = scaler.fit_transform(df[numeric_cols])

print("Final shape:", df.shape)
df.to_csv(f"{PROCESSED_DIR}/data_core_cleaned.csv", index=False)
df.head()


## 5. Exploratory Data Analysis — real dataset

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="Purples", fmt=".2f")
plt.title("Correlation between numeric features")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=df, x="Soil Type", ax=axes[0])
axes[0].set_title("Soil Type distribution")
axes[0].tick_params(axis="x", rotation=30)

sns.countplot(data=df, x="Fertilizer Name", ax=axes[1])
axes[1].set_title("Fertilizer Name distribution (target)")
axes[1].tick_params(axis="x", rotation=45)

sns.countplot(data=df, y="Crop Type", ax=axes[2], order=df["Crop Type"].value_counts().index)
axes[2].set_title("Crop Type distribution")
plt.tight_layout()
plt.show()


## Notes
- **Structured data (Section 4-5) is real** — `data_core.csv`, 8,000 rows. It has no missing values or duplicates, so the cleaning steps mostly validate rather than repair. It also has no pH or Organic matter columns, so soil-health diagnosis will need an additional data source for those two parameters.
- **Image data (Section 2-3) is still placeholder** — delete Section 2's generator once a real labeled soil-image dataset is sourced, point `RAW_IMG_DIR` at it, and re-run Section 3.
- `remove_background` uses simple Otsu thresholding — fine for prototyping; consider a trained segmentation model later for cluttered field photos.
- `valid_ranges` in Section 4 are reasonable placeholders — tighten them to real agronomic bounds for your target region/crops before this feeds Milestone 2 model training.